## Three-Stream Adaptive ST-GCN
Notebook đã được tách thành các cell nhỏ theo pipeline để dễ đọc, kiểm tra dữ liệu và chạy lại từng phần.


## Imports And Config


In [ ]:
"""
train_mydata_multiseed.py
=========================
Three-Stream Adaptive STGCN — Multi-Seed Stability Evaluation

FIXES APPLIED:
  [FIX-A] mixup_data: ưu tiên mix cùng class (same-label mixup) để tránh
          làm mờ decision boundary khi dữ liệu mất cân bằng.
  [FIX-B] FocalLoss alpha_w: đồng bộ với sampler — dùng weight theo 4
          categories thay vì gộp binary, giúp loss và sampler nhất quán.
  [FIX-C] build_weighted_sampler: thêm log tỷ lệ flip/base trong train set
          để theo dõi skew do flip 100% vào train.
"""
import copy
import json
import math
import os
import random
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (accuracy_score, auc, confusion_matrix,
                             precision_recall_curve, precision_recall_fscore_support,
                             roc_auc_score, roc_curve)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

warnings.filterwarnings('ignore')

SEEDS      = [42, 123, 2026, 777, 31415]
SPLIT_SEED = SEEDS[0]   # Cố định split để ensemble có cùng test set

ALL_NPZ         = "/kaggle/input/datasets/toannv2020/mydata-fall-v2/custom_all.npz"
BASE_RESULTS    = "results"

FALL_CATEGORIES = ["Lying_Falls", "Sitting_Falls", "Standing_Falls"]
ALL_CATEGORIES  = ["non_fall"] + FALL_CATEGORIES

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

NUM_CLASSES   = 2
IN_CHANNELS   = 3
NUM_KEYPOINTS = 14
SEQ_LEN       = 60

CATEGORY_LABEL = {
    'non_fall':       0,
    'Lying_Falls':    1,
    'Sitting_Falls':  2,
    'Standing_Falls': 3,
}

STREAM_CHANNELS = [24, 24, 24, 48, 48, 48, 96, 96, 96]
STREAM_STRIDES  = [ 1,  1,  1,  2,  1,  1,   2,   1,   1]
TCN_KERNEL      = 9
SKIP_BLOCK_IDX  = 2
DROPOUT         = 0.75
DROP_PATH_RATE  = 0.18
CONF_THRESH     = 0.5
FOCAL_GAMMA     = 0.85
LABEL_SMOOTHING = 0.16
FLIP_PAIRS      = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12)]

BATCH_SIZE       = 32
EPOCHS           = 120
LEARNING_RATE    = 1.2e-4
WEIGHT_DECAY     = 2.0e-2
WARMUP_EPOCHS    = 10
MIN_EPOCHS       = 30
PLATEAU_PATIENCE = 12
PLATEAU_DELTA    = 1e-4
MIXUP_ALPHA      = 0.35
INFERENCE_THR    = 0.65
FALL_ALPHA_BOOST = 0.85
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ══════════════════════════════════════════════════════════════════════════════
# INPUT LOADING / INSPECTION
# ══════════════════════════════════════════════════════════════════════════════
def load_training_data(npz_path=ALL_NPZ):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"Missing: {npz_path}")

    raw = np.load(npz_path, allow_pickle=True)
    required_keys = ["X_joint", "y", "img_w", "img_h", "subject", "is_flip", "category"]
    missing = [key for key in required_keys if key not in raw.files]
    if missing:
        raise KeyError(f"Missing keys in {npz_path}: {missing}")

    return dict(
        X_joint=raw["X_joint"].astype(np.float32),
        y=raw["y"].astype(np.int64),
        img_w=raw["img_w"].astype(np.int32),
        img_h=raw["img_h"].astype(np.int32),
        subject=raw["subject"],
        is_flip=raw["is_flip"].astype(np.int32),
        category=raw["category"],
    )


def inspect_input_file(npz_path=ALL_NPZ):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"Missing: {npz_path}")

    raw = np.load(npz_path, allow_pickle=True)
    print(f"Input file: {npz_path}")
    print(f"Keys: {raw.files}")
    for key in raw.files:
        arr = raw[key]
        print(f"  {key:<10} shape={arr.shape} dtype={arr.dtype}")

    if "X_joint" in raw.files:
        expected = (SEQ_LEN, NUM_KEYPOINTS, IN_CHANNELS)
        actual = tuple(raw["X_joint"].shape[1:])
        if actual != expected:
            raise ValueError(f"X_joint sample shape must be {expected}, got {actual}")
        print(f"X_joint sample shape OK: {actual}")

    if "y" in raw.files:
        y = raw["y"].astype(np.int64)
        print(f"Samples: {len(y)} | fall={int(y.sum())} | non_fall={int((y == 0).sum())}")

    return raw.files


def print_input_summary(data):
    log(
        f"Loaded: {len(data['y'])} samples  "
        f"(Fall={int(data['y'].sum())}  NF={int((data['y'] == 0).sum())})"
    )
    log(f"X_joint shape: {data['X_joint'].shape}  dtype={data['X_joint'].dtype}")


## Inspect Input NPZ


In [ ]:
inspect_input_file(ALL_NPZ)


## Utilities


In [ ]:
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)


def log(msg, level="INFO"):
    tag = {"INFO": "ℹ️ ", "GOOD": "✅", "WARN": "⚠️ ", "FAIL": "🔴"}.get(level, "")
    print(f"{tag} {msg}")

def sep(char="─", n=68):
    print(char * n)


## Splitting And Scheduler


In [ ]:
def split_indices(y, subject_arr, is_flip_arr, category_arr,
                  train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
                  seed=42):
    rng = np.random.default_rng(seed)
    all_subjects = sorted(np.unique(subject_arr))
    all_cats     = sorted(np.unique(category_arr))
    train_idx, val_idx, test_idx = [], [], []

    for subj in all_subjects:
        for cat in all_cats:
            mask_base = (
                (subject_arr == subj) &
                (category_arr == cat) &
                (is_flip_arr == 0)
            )
            base_ids = np.where(mask_base)[0]
            if len(base_ids) == 0:
                continue

            perm     = rng.permutation(len(base_ids))
            base_ids = base_ids[perm]

            n_total      = len(base_ids)
            n_val        = max(1, round(n_total * val_ratio))
            n_test       = max(1, round(n_total * TEST_RATIO))
            n_test       = min(n_test,  n_total - n_val - 1)
            n_val        = min(n_val,   n_total - n_test - 1)
            n_train_base = n_total - n_val - n_test

            tr_base = base_ids[:n_train_base]
            va_ids  = base_ids[n_train_base: n_train_base + n_val]
            te_ids  = base_ids[n_train_base + n_val:]

            val_idx.extend(va_ids.tolist())
            test_idx.extend(te_ids.tolist())

            mask_flip = (
                (subject_arr == subj) &
                (category_arr == cat) &
                (is_flip_arr == 1)
            )
            flip_ids = np.where(mask_flip)[0]
            train_idx.extend(tr_base.tolist())
            train_idx.extend(flip_ids.tolist())

    return (np.array(train_idx, dtype=np.int64),
            np.array(val_idx,   dtype=np.int64),
            np.array(test_idx,  dtype=np.int64))


def build_scheduler(optimizer, warmup_epochs, total_epochs):
    lr_min = 1e-6
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress  = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        cosine    = 0.5 * (1.0 + math.cos(math.pi * progress))
        min_ratio = lr_min / LEARNING_RATE
        return min_ratio + (1.0 - min_ratio) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


class FocalLoss(nn.Module):
    """
    [FIX-B] alpha nhận vào tensor 4 phần tử (theo CATEGORY_LABEL),
    sau đó map về binary khi tính loss để đồng bộ với sampler.
    Nếu alpha_mode='binary' thì giữ nguyên hành vi cũ (2 phần tử).
    """
    def __init__(self, gamma=FOCAL_GAMMA, alpha=None, reduction='mean',
                 label_smoothing=LABEL_SMOOTHING, alpha_mode='binary'):
        super().__init__()
        self.gamma      = gamma
        self.alpha      = alpha          # tensor shape (2,) — binary
        self.reduction  = reduction
        self.ls         = label_smoothing
        self.alpha_mode = alpha_mode

    def forward(self, logits, targets):
        n_cls = logits.size(1)
        log_p = F.log_softmax(logits, dim=1)
        p     = log_p.exp()
        pt    = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        fw    = (1.0 - pt) ** self.gamma
        if self.alpha is not None:
            fw = self.alpha[targets] * fw
        if self.ls > 0:
            smooth = torch.full_like(logits, self.ls / n_cls)
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls + self.ls / n_cls)
            ce = -(smooth * log_p).sum(dim=1)
        else:
            ce = -log_p.gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = fw * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()


## ST-GCN Model


In [ ]:
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__(); self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep  = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask  = torch.empty(shape, dtype=x.dtype, device=x.device).bernoulli_(keep) / keep
        return x * mask


class SkeletonGraph:
    NUM_PARTITIONS = 3
    def __init__(self, num_nodes=NUM_KEYPOINTS):
        self.num_nodes = num_nodes
        self.neighbor_link = [
            (0,1),(0,2),(1,2),(1,3),(3,5),(2,4),(4,6),
            (1,7),(2,8),(7,8),(7,9),(9,11),(8,10),(10,12),
            (7,13),(8,13),
        ]
        self.HIP_DIST = {13:0,7:1,8:1,1:2,2:2,9:2,10:2,
                         0:3,3:3,4:3,11:3,12:3,5:4,6:4}
        self.A = self._build_adjacency()

    def _build_adjacency(self):
        K, V = self.NUM_PARTITIONS, self.num_nodes
        A    = np.zeros((K, V, V), dtype=np.float32)
        for i in range(V): A[0,i,i] = 1.0
        for i, j in self.neighbor_link:
            di, dj = self.HIP_DIST[i], self.HIP_DIST[j]
            if   di < dj: A[1,j,i]=1; A[2,i,j]=1
            elif dj < di: A[1,i,j]=1; A[2,j,i]=1
            else:         A[1,i,j]=A[1,j,i]=1
        for k in range(K): A[k] = self._sym_norm(A[k])
        return A

    @staticmethod
    def _sym_norm(A):
        A = A + np.eye(A.shape[0])
        d = A.sum(axis=1)
        d_inv = np.where(d > 0, d**-0.5, 0.0)
        D = np.diag(d_inv)
        return D @ A @ D


class AdaptiveSpatialGraphConv(nn.Module):
    def __init__(self, in_ch, out_ch, num_nodes, num_partitions):
        super().__init__()
        self.K       = num_partitions
        self.conv    = nn.Conv2d(in_ch, out_ch * num_partitions, kernel_size=1)
        self.A_adapt = nn.Parameter(torch.zeros(num_partitions, num_nodes, num_nodes))

    def forward(self, x, A_static):
        A_eff = A_static + torch.tanh(self.A_adapt)
        N,_,T,V = x.shape
        x = self.conv(x).view(N, self.K, -1, T, V)
        return torch.einsum('kvw,nkctv->nctw', A_eff, x)


class TemporalConv(nn.Module):
    def __init__(self, channels, kernel_size=9, stride=1):
        super().__init__()
        pad = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(channels, channels, (kernel_size,1),(stride,1),(pad,0))
        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.tmp_maxpool = nn.MaxPool2d((kernel_size,1),(stride,1),(pad,0))

    def forward(self, x):
        res = self.tmp_maxpool(x)
        x   = self.relu(self.bn(self.conv(x)))
        return x + res


class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, num_nodes, num_partitions, stride=1, drop_path_rate=0.0):
        super().__init__()
        self.gcn      = AdaptiveSpatialGraphConv(in_ch, out_ch, num_nodes, num_partitions)
        self.gcn_bn   = nn.BatchNorm2d(out_ch)
        self.res_conv = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch))
                         if in_ch != out_ch else nn.Identity())
        self.sp_pool  = nn.MaxPool2d((1,3),(1,1),(0,1))
        self.tcn      = TemporalConv(out_ch, TCN_KERNEL, stride)
        self.drop_path = DropPath(drop_path_rate)
        self.relu     = nn.ReLU(inplace=True)

    def forward(self, x, A):
        x_gcn = self.gcn_bn(self.gcn(x, A))
        x_res = self.relu(self.drop_path(x_gcn) + self.res_conv(x))
        x_sp  = self.sp_pool(x_res)
        return self.relu(self.drop_path(self.tcn(x_sp)))


class StreamEncoder(nn.Module):
    def __init__(self, in_channels, adjacency, channels, strides):
        super().__init__()
        K, V, _ = adjacency.shape
        self.register_buffer('A', torch.tensor(adjacency, dtype=torch.float32))
        self.input_bn   = nn.BatchNorm2d(in_channels)
        self.input_proj = nn.Sequential(
            nn.Conv2d(in_channels, channels[0], 1),
            nn.BatchNorm2d(channels[0]), nn.ReLU(inplace=True))
        n_blocks, blocks, prev = len(channels), [], channels[0]
        for i, (ch, st) in enumerate(zip(channels, strides)):
            dpr = DROP_PATH_RATE * i / max(1, n_blocks - 1)
            blocks.append(STGCNBlock(prev, ch, V, K, st, drop_path_rate=dpr))
            prev = ch
        self.blocks      = nn.ModuleList(blocks)
        self.edge_importance = nn.ParameterList([
            nn.Parameter(torch.ones(K, V, V)) for _ in self.blocks
        ])
        self.spatial_gap = nn.AdaptiveAvgPool2d((None, 1))

    def forward(self, x):
        x = self.input_proj(self.input_bn(x))
        early = None
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.A * self.edge_importance[i])
            if i == SKIP_BLOCK_IDX:
                early = x
        return self.spatial_gap(x).squeeze(-1), early


class CrossStreamSkip(nn.Module):
    def __init__(self, early_ch, final_ch):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(early_ch, final_ch, 1),
                                  nn.BatchNorm2d(final_ch), nn.ReLU(inplace=True))
        self.spatial_gap = nn.AdaptiveAvgPool2d((None, 1))

    def forward(self, ej, em, ea):
        return self.spatial_gap(self.proj(ej + em + ea)).squeeze(-1)


class ConfidenceMasking(nn.Module):
    def __init__(self, conf_threshold=CONF_THRESH):
        super().__init__(); self.threshold = conf_threshold

    def forward(self, feats, x_joint):
        T_prime = feats[0].shape[-1]
        conf    = x_joint[:, 2, :, :].mean(dim=-1)
        conf_ds = F.adaptive_avg_pool1d(conf.unsqueeze(1).float(), T_prime).squeeze(1)
        mask    = (conf_ds > self.threshold).float().unsqueeze(1)
        return [f * mask for f in feats], mask


class ClassificationModule(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, num_classes=2, dropout=0.5):
        super().__init__()
        self.fc1    = nn.Conv1d(in_dim, hidden_dim, 1)
        self.lrelu1 = nn.LeakyReLU(0.1, inplace=True)
        self.ln     = nn.LayerNorm(hidden_dim)
        self.lrelu2 = nn.LeakyReLU(0.1, inplace=True)
        self.drop   = nn.Dropout(p=dropout)
        self.fc_out = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x     = self.lrelu1(self.fc1(x))
        x     = self.lrelu2(self.ln(x.permute(0,2,1)).permute(0,2,1))
        x     = self.drop(x)
        x_avg = x.mean(dim=-1)
        return self.fc_out(x_avg)


class ThreeStreamSTGCN(nn.Module):
    def __init__(self, in_channels=IN_CHANNELS, num_classes=NUM_CLASSES,
                 skeleton_adj=None, channels=STREAM_CHANNELS,
                 strides=STREAM_STRIDES, dropout=DROPOUT):
        super().__init__()
        if skeleton_adj is None:
            skeleton_adj = SkeletonGraph().A
        self.stream_joint  = StreamEncoder(in_channels, skeleton_adj, channels, strides)
        self.stream_motion = StreamEncoder(in_channels, skeleton_adj, channels, strides)
        self.stream_angle  = StreamEncoder(in_channels, skeleton_adj, channels, strides)
        early_ch = channels[SKIP_BLOCK_IDX]; final_ch = channels[-1]
        self.cross_skip = CrossStreamSkip(early_ch, final_ch)
        self.classifier = ClassificationModule(4 * final_ch, 128, num_classes, dropout)

    def forward(self, x_joint, x_motion, x_angle):
        f1, ej = self.stream_joint(x_joint)
        f2, em = self.stream_motion(x_motion)
        f3, ea = self.stream_angle(x_angle)
        T_prime = f1.shape[-1]
        f_skip  = F.adaptive_avg_pool1d(self.cross_skip(ej, em, ea), T_prime)
        feat = torch.cat([f1, f2, f3, f_skip], dim=1)
        return self.classifier(feat)


## Data Helpers And Dataset


In [ ]:
PARENT_MAP_AUG = {
    13:None, 7:13, 8:13, 1:7, 2:8, 0:1,
    3:1, 4:2, 5:3, 6:4, 9:7, 10:8, 11:9, 12:10,
}

def compute_motion_np(j):
    m = np.zeros_like(j); m[1:] = j[1:]-j[:-1]; m[:,:,2]=j[:,:,2]; return m

def compute_angle_np(j):
    F,V,_ = j.shape; angle = np.zeros((F,V,3),dtype=np.float32)
    for v in range(V):
        p = PARENT_MAP_AUG[v]
        if p is None:
            angle[:,v,0]=1.0; angle[:,v,2]=j[:,v,2]
        else:
            dy=j[:,v,1]-j[:,p,1]; dx=j[:,v,0]-j[:,p,0]
            th=np.arctan2(dy,dx)
            angle[:,v,0]=np.cos(th); angle[:,v,1]=np.sin(th)
            angle[:,v,2]=np.minimum(j[:,v,2],j[:,p,2])
    return angle

def aug_rotate(j, deg):
    theta=deg*np.pi/180; c,s=np.cos(theta),np.sin(theta)
    R=np.array([[c,-s],[s,c]],dtype=np.float32)
    j=j.copy(); pivot=j[:,13:14,:2]
    j[:,:,:2]=(j[:,:,:2]-pivot)@R.T+pivot; return j

def aug_zoom(j, factor):
    j=j.copy(); pivot=j[:,13:14,:2]
    j[:,:,:2]=(j[:,:,:2]-pivot)*factor+pivot; return j

def aug_mask_joints(j, mask_prob=0.15):
    j=j.copy(); mask=np.random.rand(j.shape[1])<mask_prob
    j[:,mask,2]=0.0; return j

def aug_hflip(j, img_w):
    j=j.copy(); j[:,:,0]=img_w-j[:,:,0]
    for l,r in FLIP_PAIRS: j[:,[l,r],:]=j[:,[r,l],:]; return j

def aug_noise(j, img_w, ratio=0.005):
    j=j.copy()
    j[:,:,:2]+=np.random.normal(0,ratio*img_w,j[:,:,:2].shape).astype(np.float32); return j

def aug_speed(j, factor):
    F=len(j); new_F=max(8,int(F/factor))
    ot=np.linspace(0,1,F); nt=np.linspace(0,1,new_F)
    out=np.zeros((new_F,j.shape[1],3),dtype=np.float32)
    for v in range(j.shape[1]):
        for c in range(3): out[:,v,c]=np.interp(nt,ot,j[:,v,c])
    if new_F>=SEQ_LEN: return out[:SEQ_LEN]
    pad=np.zeros((SEQ_LEN-new_F,j.shape[1],3),dtype=np.float32)
    return np.concatenate([out,pad])

def aug_crop(j, img_w, img_h, ratio=0.85):
    j=j.copy(); wc=int(img_w*ratio); hc=int(img_h*ratio)
    x0=np.random.randint(0,max(1,img_w-wc)); y0=np.random.randint(0,max(1,img_h-hc))
    in_box=((j[:,:,0]>=x0)&(j[:,:,0]<x0+wc)&(j[:,:,1]>=y0)&(j[:,:,1]<y0+hc))
    j[:,:,2]=np.where(in_box,j[:,:,2],0.0)
    j[:,:,0]=np.clip(j[:,:,0]-x0,0,wc-1); j[:,:,1]=np.clip(j[:,:,1]-y0,0,hc-1); return j

def augment_online(j, img_w, img_h, label):
    is_fall = (label == 1)
    if np.random.rand() < (0.55 if is_fall else 0.50):
        j = aug_mask_joints(j, np.random.uniform(0.05, 0.22 if is_fall else 0.18))
    if np.random.rand() < (0.75 if is_fall else 0.65):
        j = aug_rotate(j, np.random.uniform(-25 if is_fall else -20, 25 if is_fall else 20))
    if np.random.rand() < (0.70 if is_fall else 0.60):
        lo,hi = (0.78,1.22) if is_fall else (0.82,1.18)
        j = aug_zoom(j, np.random.uniform(lo, hi))
    if np.random.rand() < 0.5:
        j = aug_hflip(j, img_w)
    if np.random.rand() < (0.45 if is_fall else 0.40):
        j = aug_noise(j, img_w, np.random.uniform(0.003, 0.014 if is_fall else 0.010))
    if np.random.rand() < (0.45 if is_fall else 0.25):
        j = aug_speed(j, np.random.uniform(0.70,1.35) if is_fall else np.random.uniform(0.85,1.15))
    if np.random.rand() < (0.35 if is_fall else 0.20):
        j = aug_crop(j, img_w, img_h, np.random.uniform(0.80, 0.95))
    return j


class CustomFallDataset(Dataset):
    def __init__(self, X_joint, y, img_w, img_h, category, indices, augment=False):
        self.X_joint  = X_joint
        self.y        = y
        self.img_w    = img_w
        self.img_h    = img_h
        self.category = category
        self.indices  = indices
        self.augment  = augment

    def get_labels(self):
        return self.y[self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        gi  = self.indices[idx]
        j   = self.X_joint[gi].copy()
        y   = int(self.y[gi])
        iw  = int(self.img_w[gi])
        ih  = int(self.img_h[gi])
        cat = str(self.category[gi])

        if self.augment:
            j = augment_online(j, iw, ih, y)

        m = compute_motion_np(j)
        a = compute_angle_np(j)

        j = torch.FloatTensor(j.transpose(2, 0, 1))
        m = torch.FloatTensor(m.transpose(2, 0, 1))
        a = torch.FloatTensor(a.transpose(2, 0, 1))

        cat_label = CATEGORY_LABEL.get(cat, 0)
        return j, m, a, torch.tensor(y, dtype=torch.long), cat, cat_label


def build_weighted_sampler(dataset):
    labels      = dataset.get_labels().astype(np.int64)
    categories  = dataset.category[dataset.indices]
    cat_indices = np.array([CATEGORY_LABEL.get(c, 0) for c in categories])
    n_cats      = len(CATEGORY_LABEL)
    cat_counts  = np.bincount(cat_indices, minlength=n_cats)

    # [FIX] đồng nhất sqrt inverse với FocalLoss alpha
    cat_counts_safe = np.where(cat_counts == 0, 1, cat_counts)
    weight_per_cat  = 1.0 / np.sqrt(cat_counts_safe.astype(np.float64))
    sample_weights  = torch.DoubleTensor([weight_per_cat[ci] for ci in cat_indices])

    total      = len(labels)
    fall_total = int(labels.sum())
    log(f"  Train category dist: " +
        " | ".join(f"{k}={cat_counts[v]}" for k, v in CATEGORY_LABEL.items()))
    log(f"  Train binary: fall={fall_total} ({100*fall_total/total:.1f}%)  "
        f"non_fall={total-fall_total} ({100*(total-fall_total)/total:.1f}%)")
    log(f"  Sampler weights (sqrt-inv): " +
        " | ".join(f"{k}={weight_per_cat[v]:.4f}" for k, v in CATEGORY_LABEL.items()))

    return WeightedRandomSampler(sample_weights, len(labels), replacement=True)


# ── [FIX-A] Same-label Mixup ──────────────────────────────────────────────────
def mixup_data(j, m, a, y, alpha=MIXUP_ALPHA):
    """
    Ưu tiên mix cùng class (same-label mixup).
    Khi dữ liệu mất cân bằng, cross-class mixup tạo ra vùng ambiguous
    gần decision boundary, làm mô hình khó học fall minority class.
    Same-label mixup tăng diversity mà không làm mờ boundary.

    Fallback: nếu class chỉ có 1 sample trong batch → dùng random như cũ.
    """
    if alpha <= 0 or np.random.rand() > 0.5:
        return j, m, a, y, None

    lam = float(np.random.beta(alpha, alpha))
    B   = j.size(0)

    # Xây dựng chỉ số mix ưu tiên cùng class
    same_class_idx = torch.zeros(B, dtype=torch.long, device=j.device)
    for i in range(B):
        yi         = y[i].item()
        candidates = (y == yi).nonzero(as_tuple=True)[0]
        # Loại bỏ chính nó nếu có đủ candidate
        candidates = candidates[candidates != i]
        if len(candidates) > 0:
            pick = candidates[torch.randint(len(candidates), (1,)).item()]
        else:
            # Fallback: random bất kỳ (batch quá nhỏ hoặc class duy nhất)
            pool = torch.arange(B, device=j.device)
            pool = pool[pool != i]
            pick = pool[torch.randint(len(pool), (1,)).item()] if len(pool) > 0 else i
        same_class_idx[i] = pick

    idx = same_class_idx
    j_mix = lam * j + (1 - lam) * j[idx]
    m_mix = lam * m + (1 - lam) * m[idx]
    a_mix = lam * a + (1 - lam) * a[idx]

    return j_mix, m_mix, a_mix, y, (y[idx], lam)
# ─────────────────────────────────────────────────────────────────────────────


## Training And Evaluation


In [ ]:
def mixup_criterion(criterion, logits, y, mixed_info):
    if mixed_info is None:
        return criterion(logits, y)
    y_b, lam = mixed_info
    return lam*criterion(logits, y) + (1-lam)*criterion(logits, y_b)


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0; preds_all = []; labels_all = []
    for j, m, a, y, _cat, _cl in loader:
        j, m, a, y = j.to(device), m.to(device), a.to(device), y.to(device)
        j, m, a, y, mi = mixup_data(j, m, a, y)
        optimizer.zero_grad()
        logits = model(j, m, a)
        loss   = mixup_criterion(criterion, logits, y, mi)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    acc = accuracy_score(labels_all, preds_all)
    pr_t, rc_t, _, _ = precision_recall_fscore_support(
        labels_all, preds_all, average='binary', zero_division=0)
    denom_f2 = 4 * pr_t + rc_t
    f2 = (5 * pr_t * rc_t / denom_f2) if denom_f2 > 0 else 0.0
    return total_loss / len(loader), acc, f2


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0; preds_all = []; labels_all = []; probs_all = []
    cats_all = []; cat_labels_all = []
    for j, m, a, y, cat, cat_label in loader:
        j, m, a, y = j.to(device), m.to(device), a.to(device), y.to(device)
        logits = model(j, m, a)
        total_loss += criterion(logits, y).item()
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
        probs_all.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
        cats_all.extend(list(cat))
        cat_labels_all.extend(cat_label.numpy().tolist())
    acc = accuracy_score(labels_all, preds_all)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels_all, preds_all, average='binary', zero_division=0)
    cm = confusion_matrix(labels_all, preds_all)
    return (total_loss / len(loader), acc, prec, rec, f1,
            cm, preds_all, labels_all, probs_all, cats_all, cat_labels_all)


def eval_at_threshold(labels, probs, thr):
    preds  = (np.array(probs)  >= thr).astype(int)
    labels = np.array(labels)
    if len(np.unique(labels)) < 2:
        tp = int(((preds==1)&(labels==1)).sum()); fp = int(((preds==1)&(labels==0)).sum())
        fn = int(((preds==0)&(labels==1)).sum()); tn = int(((preds==0)&(labels==0)).sum())
    else:
        tn, fp, fn, tp = (int(v) for v in confusion_matrix(labels, preds).ravel())
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    return dict(acc=(tp+tn)/(tp+tn+fp+fn), prec=prec, rec=rec,
                spec=spec, f1=f1, mr=1-rec,
                fpr=fp/(fp+tn) if (fp+tn)>0 else 0.0,
                tp=tp, fp=fp, fn=fn, tn=tn)


def eval_per_category(labels_all, probs_all, cats_all, thr=0.5):
    """
    One-vs-rest confusion cho 4 category:
      non_fall, Lying_Falls, Sitting_Falls, Standing_Falls

    Lưu ý:
    - Model là binary (fall / non_fall), nên với 3 fall-category:
      pred_positive được hiểu là (pred == fall).
    """
    probs_arr = np.array(probs_all)
    cats_arr  = np.array(cats_all)
    preds_arr = (probs_arr >= thr).astype(int)  # 1=fall, 0=non_fall

    results = {}
    for cat in ALL_CATEGORIES:
        true_pos = (cats_arr == cat)

        if cat == "non_fall":
            pred_pos = (preds_arr == 0)  # dự đoán non_fall
        else:
            pred_pos = (preds_arr == 1)  # dự đoán fall (không phân subtype)

        tp = int(( pred_pos &  true_pos).sum())
        fp = int(( pred_pos & ~true_pos).sum())
        fn = int((~pred_pos &  true_pos).sum())
        tn = int((~pred_pos & ~true_pos).sum())

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

        results[cat] = dict(
            precision=prec,
            rec=rec,
            spec=spec,
            f1=f1,
            mr=1 - rec,
            tp=tp, fp=fp, fn=fn, tn=tn,
            n=int(true_pos.sum()),
            support=int(true_pos.sum()),
        )

    return results


## Single Seed Run


In [ ]:
def run_one_seed(seed: int, data: dict, save_dir: str, split_seed: int = None) -> dict:
    os.makedirs(save_dir, exist_ok=True)
    set_global_seed(seed)

    X_joint     = data['X_joint']
    y           = data['y']
    img_w       = data['img_w']
    img_h       = data['img_h']
    subject_arr = data['subject']
    is_flip_arr = data['is_flip']
    category    = data['category']

    _split_seed = split_seed if split_seed is not None else seed
    train_idx, val_idx, test_idx = split_indices(
        y, subject_arr, is_flip_arr, category, seed=_split_seed)

    train_ds = CustomFallDataset(X_joint, y, img_w, img_h, category, train_idx, augment=True)
    val_ds   = CustomFallDataset(X_joint, y, img_w, img_h, category, val_idx,   augment=False)
    test_ds  = CustomFallDataset(X_joint, y, img_w, img_h, category, test_idx,  augment=False)

    cat_labels_train = np.array([CATEGORY_LABEL[c] for c in category[train_idx]])
    cat_counts_4     = np.bincount(cat_labels_train, minlength=4).astype(np.float32)
    w4               = 1.0 / np.sqrt(cat_counts_4.clip(min=1))

    fall_cat_counts  = cat_counts_4[1:].clip(min=1)
    fall_w_weighted  = float(np.average(w4[1:], weights=fall_cat_counts))
    fall_w           = fall_w_weighted * FALL_ALPHA_BOOST
    nfall_w          = float(w4[0])

    alpha_w = torch.FloatTensor([nfall_w, fall_w])
    alpha_w = (alpha_w / alpha_w.sum()).to(DEVICE)

    log(f"  alpha_w → non_fall={alpha_w[0]:.4f}  fall={alpha_w[1]:.4f}  "
        f"(sqrt-inv w4: {w4.round(4)})")

    sampler      = build_weighted_sampler(train_ds)
    train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    model     = ThreeStreamSTGCN().to(DEVICE)
    criterion = FocalLoss(FOCAL_GAMMA, alpha_w, label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = build_scheduler(optimizer, WARMUP_EPOCHS, EPOCHS)

    best_loss = float('inf'); best_va_f2 = -1.0; plateau_cnt = 0; best_ep = 1
    model_path = os.path.join(save_dir, "best_model.pth")

    # ── [NEW] Lịch sử epoch để vẽ convergence chart ──────────────────────────
    history = dict(
        epoch       = [],
        train_loss  = [],
        val_loss    = [],
        train_f2    = [],
        val_f2      = [],
    )
    # ─────────────────────────────────────────────────────────────────────────

    for epoch in range(EPOCHS):
        tr_loss, _, tr_f2 = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        scheduler.step()
        va_loss, _, va_prec_es, va_rec_es, _, *_ = evaluate(model, val_loader, criterion, DEVICE)
        _denom = 4 * va_prec_es + va_rec_es
        va_f2  = (5 * va_prec_es * va_rec_es / _denom) if _denom > 0 else 0.0

        # ── [NEW] Ghi lịch sử ───────────────────────────────────────────────
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(float(tr_loss))
        history['val_loss'].append(float(va_loss))
        history['train_f2'].append(float(tr_f2))
        history['val_f2'].append(float(va_f2))
        # ────────────────────────────────────────────────────────────────────

        improved = ((va_loss < best_loss - PLATEAU_DELTA) or
                    (va_f2   > best_va_f2 + PLATEAU_DELTA))
        if improved:
            best_loss = va_loss; best_va_f2 = va_f2
            plateau_cnt = 0;     best_ep = epoch + 1
            torch.save(model.state_dict(), model_path)
        else:
            plateau_cnt += 1

        if (epoch + 1) % 25 == 0:
            print(f"    ep{epoch+1:>3}  tr_f2={tr_f2:.4f}  va_f2={va_f2:.4f}")

        if epoch + 1 >= MIN_EPOCHS and plateau_cnt >= PLATEAU_PATIENCE:
            log(f"    Early stop @ ep{epoch+1}  (best={best_ep})")
            break

    model.load_state_dict(torch.load(model_path, weights_only=True))

    # ── ROC Youden Index — chọn threshold tối ưu trên val set ────────────────
    (_, _, _, _, _, _, _, va_labels_yd, va_probs_yd, _, _) = evaluate(
        model, val_loader, criterion, DEVICE)
    if len(set(va_labels_yd)) > 1:
        fpr_val, tpr_val, thr_val = roc_curve(va_labels_yd, va_probs_yd)
        youden_j    = tpr_val - fpr_val
        best_idx    = int(np.argmax(youden_j))
        optimal_thr = float(thr_val[best_idx])
        log(f"  Youden threshold (val): {optimal_thr:.4f}  "
            f"(Sensitivity={tpr_val[best_idx]:.4f}  "
            f"Specificity={1-fpr_val[best_idx]:.4f})")
    else:
        optimal_thr = INFERENCE_THR
        log(f"  Val set single class → fallback thr={optimal_thr:.4f}", "WARN")
    # ─────────────────────────────────────────────────────────────────────────

    (_, t_acc, t_prec, t_rec, t_f1, t_cm,
     t_preds, t_labels, t_probs, t_cats, _) = evaluate(model, test_loader, criterion, DEVICE)

    roc_auc = roc_auc_score(t_labels, t_probs) if len(set(t_labels)) > 1 else 0.0
    pc, rc, _ = precision_recall_curve(t_labels, t_probs)
    pr_auc    = auc(rc, pc) if len(set(t_labels)) > 1 else 0.0

    res         = eval_at_threshold(t_labels, t_probs, optimal_thr)
    cat_metrics = eval_per_category(t_labels, t_probs, t_cats, optimal_thr)

    t_acc  = res['acc']
    t_prec = res['prec']
    t_rec  = res['rec']
    t_f1   = res['f1']

    cat_recall = {}
    for cat in FALL_CATEGORIES:
        m = cat_metrics.get(cat)
        cat_recall[cat] = m['rec'] if m else float('nan')

    t_preds_thr = (np.array(t_probs) >= optimal_thr).astype(int)
    prec_per, rec_per, f1_per, sup_per = precision_recall_fscore_support(
        t_labels, t_preds_thr, labels=[0, 1], zero_division=0)
    cm_thr = confusion_matrix(t_labels, t_preds_thr).tolist()
        # confusion matrix dạng [[tn, fp], [fn, tp]]
    tn, fp = cm_thr[0]
    fn, tp = cm_thr[1]

    confusion_by_class = {
        "non_fall": {
            "tp": int(tn),
            "fp": int(fn),
            "fn": int(fp),
            "tn": int(tp),
        },
        "fall": {
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "tn": int(tn),
        }
    }


    result = dict(
        seed     = seed,
        best_ep  = best_ep,
        acc      = float(t_acc),
        prec     = float(t_prec),
        rec      = float(t_rec),
        spec     = float(res['spec']),
        f1       = float(t_f1),
        mr       = float(res['mr']),
        roc_auc  = float(roc_auc),
        pr_auc   = float(pr_auc),
        tp=res['tp'], fp=res['fp'], fn=res['fn'], tn=res['tn'],
        optimal_thr = optimal_thr,
        t_probs_list  = [float(p) for p in t_probs],
        t_labels_list = [int(l)   for l in t_labels],
        t_cats_list   = list(t_cats),
        cat_recall=cat_recall,
        cat_metrics=cat_metrics,
        per_class_metrics=dict(
            non_fall=dict(
                precision = float(prec_per[0]),
                recall    = float(rec_per[0]),
                f1        = float(f1_per[0]),
                support   = int(sup_per[0]),
            ),
            fall=dict(
                precision = float(prec_per[1]),
                recall    = float(rec_per[1]),
                f1        = float(f1_per[1]),
                support   = int(sup_per[1]),
            ),
        ),
        confusion_matrix = cm_thr,
        confusion_by_class=confusion_by_class,
        history          = history,   # [NEW]
    )
    with open(os.path.join(save_dir, "metrics.json"), "w") as f:
        json.dump(result, f, indent=2)

    return result


## Reporting And Main


In [ ]:
def plot_stability_boxplot(all_results: list, save_dir: str):
    metrics = ['acc', 'rec', 'spec', 'f1', 'roc_auc', 'pr_auc', 'mr']
    labels  = ['Accuracy', 'Sensitivity\n(Recall)', 'Specificity', 'F1-Score',
               'ROC-AUC', 'PR-AUC', 'Miss Rate']

    data_by_metric = {m: [r[m] for r in all_results] for m in metrics}
    seeds_list     = [r['seed'] for r in all_results]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(
        f"Stability Analysis — {len(all_results)} seeds: {seeds_list}\n"
        f"(Each point = one seed run, thr=Youden per-seed)",
        fontsize=13, fontweight='bold')
    axes = axes.flatten()

    colors = ['#2171b5', '#e07b39', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']

    for ax, m, lbl, col in zip(axes, metrics, labels, colors):
        vals = data_by_metric[m]
        bp   = ax.boxplot(vals, patch_artist=True, widths=0.4,
                          medianprops=dict(color='black', lw=2))
        bp['boxes'][0].set_facecolor(col + '60')
        bp['boxes'][0].set_edgecolor(col)

        jitter = np.random.default_rng(0).uniform(-0.07, 0.07, len(vals))
        for i, (v, s, jit) in enumerate(zip(vals, seeds_list, jitter)):
            ax.scatter(1 + jit, v, color=col, s=60, zorder=5)
            ax.annotate(f"s{s}", (1 + jit, v), textcoords="offset points",
                        xytext=(5, 2), fontsize=7, color='#333')

        mu, sd = np.mean(vals), np.std(vals)
        ax.set_title(f"{lbl}\n{mu:.4f} ± {sd:.4f}", fontsize=10, fontweight='bold')
        ax.set_xticks([])
        ax.set_ylim(max(0, min(vals) - 0.05), min(1.02, max(vals) + 0.05))
        ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

    axes[-1].axis('off')
    plt.tight_layout()
    path = os.path.join(save_dir, 'stability_boxplot.png')
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)
    print(f"    Saved → {path}")


def plot_stability_per_category(all_results: list, save_dir: str):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    fig.suptitle("Per-Category Fall Recall — Stability across seeds",
                 fontsize=13, fontweight='bold')
    colors = ['#2171b5', '#e07b39', '#2ca02c']
    seeds_list = [r['seed'] for r in all_results]

    for ax, cat, col in zip(axes, FALL_CATEGORIES, colors):
        vals = [r['cat_recall'].get(cat, float('nan')) for r in all_results]
        vals_clean = [v for v in vals if not np.isnan(v)]
        if not vals_clean:
            ax.set_title(f"{cat}\n(no data)"); ax.axis('off'); continue

        bp = ax.boxplot(vals_clean, patch_artist=True, widths=0.4,
                        medianprops=dict(color='black', lw=2))
        bp['boxes'][0].set_facecolor(col + '60')
        bp['boxes'][0].set_edgecolor(col)

        jitter = np.random.default_rng(1).uniform(-0.07, 0.07, len(vals))
        for v, s, jit in zip(vals, seeds_list, jitter):
            if np.isnan(v): continue
            ax.scatter(1 + jit, v, color=col, s=60, zorder=5)
            ax.annotate(f"s{s}", (1 + jit, v), textcoords="offset points",
                        xytext=(5, 2), fontsize=7, color='#333')

        mu, sd = np.mean(vals_clean), np.std(vals_clean)
        short  = cat.replace('_Falls', '')
        ax.set_title(f"{short}\nRecall: {mu:.4f} ± {sd:.4f}",
                     fontsize=10, fontweight='bold')
        ax.set_xticks([])
        ax.set_ylim(max(0, min(vals_clean) - 0.05), 1.05)
        ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

    plt.tight_layout()
    path = os.path.join(save_dir, 'stability_per_category.png')
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)
    print(f"    Saved → {path}")


def plot_seed_comparison_bar(all_results: list, save_dir: str):
    metrics = ['acc', 'rec', 'spec', 'f1', 'roc_auc']
    m_labels = ['Accuracy', 'Sensitivity', 'Specificity', 'F1', 'ROC-AUC']
    seeds    = [r['seed'] for r in all_results]
    x        = np.arange(len(metrics))
    width    = 0.8 / len(seeds)

    fig, ax = plt.subplots(figsize=(13, 5))
    cmap = plt.cm.get_cmap('tab10', len(seeds))

    for i, (res, seed) in enumerate(zip(all_results, seeds)):
        vals   = [res[m] for m in metrics]
        offset = (i - len(seeds)/2 + 0.5) * width
        bars   = ax.bar(x + offset, vals, width * 0.9, label=f"seed={seed}",
                        color=cmap(i), alpha=0.8, edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f"{v:.3f}", ha='center', va='bottom', fontsize=7, rotation=90)

    for j, m in enumerate(metrics):
        mu = np.mean([r[m] for r in all_results])
        ax.hlines(mu, j - 0.4, j + 0.4, colors='black', lw=1.5, ls='--')

    ax.set_xticks(x); ax.set_xticklabels(m_labels, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score'); ax.set_title("Per-Seed Metric Comparison (── = mean)",
                                         fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9, ncol=min(5, len(seeds)))
    ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

    plt.tight_layout()
    path = os.path.join(save_dir, 'seed_comparison_bar.png')
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)
    print(f"    Saved → {path}")


def print_stability_table(all_results: list):
    metrics = ['acc', 'rec', 'spec', 'f1', 'roc_auc', 'pr_auc', 'mr']
    m_nice  = ['Accuracy', 'Sensitivity', 'Specificity', 'F1', 'ROC-AUC', 'PR-AUC', 'MissRate']

    sep('═')
    log("STABILITY SUMMARY TABLE")
    sep('═')

    header = f"  {'Seed':>8}  " + "  ".join(f"{n:>10}" for n in m_nice)
    print(header)
    sep('-', len(header) - 2)
    for r in all_results:
        row = f"  {r['seed']:>8}  " + "  ".join(f"{r[m]:>10.4f}" for m in metrics)
        print(row)

    sep('-', len(header) - 2)
    means = {m: np.mean([r[m] for r in all_results]) for m in metrics}
    stds  = {m: np.std( [r[m] for r in all_results]) for m in metrics}
    row_m = f"  {'Mean':>8}  " + "  ".join(f"{means[m]:>10.4f}" for m in metrics)
    row_s = f"  {'Std':>8}  "  + "  ".join(f"{stds[m]:>10.4f}"  for m in metrics)
    row_cv= f"  {'CV%':>8}  "  + "  ".join(
        f"{100*stds[m]/means[m] if means[m]>1e-9 else 0:>10.2f}" for m in metrics)
    print(row_m)
    print(row_s)
    print(row_cv)
    sep('═')

    print("\n  📊 Stability Verdict (CV < 3% = stable, 3-7% = moderate, >7% = unstable):")
    for m, n in zip(metrics, m_nice):
        cv = 100 * stds[m] / means[m] if means[m] > 1e-9 else 0.0
        icon = "🟢" if cv < 3 else ("🟡" if cv < 7 else "🔴")
        print(f"    {icon}  {n:<16}: CV={cv:.2f}%  ({means[m]:.4f} ± {stds[m]:.4f})")

    sep()
    print("\n  📝 LaTeX-ready (mean ± std):")
    latex_parts = [f"{means[m]:.4f} $\\pm$ {stds[m]:.4f}" for m in metrics]
    print("  " + " & ".join(latex_parts) + " \\\\")
    sep('═')

def main():
    sep('═')
    log(f"Three-Stream Adaptive STGCN — Multi-Seed Stability  ({len(SEEDS)} seeds: {SEEDS})")
    log(f"Device: {DEVICE}  |  Batch={BATCH_SIZE}  Epochs={EPOCHS}")
    sep('═')

    data = load_training_data(ALL_NPZ)
    print_input_summary(data)
    print()

    all_results = []
    for run_i, seed in enumerate(SEEDS, 1):
        sep()
        log(f"[{run_i}/{len(SEEDS)}]  SEED = {seed}")
        sep()
        save_dir = os.path.join(BASE_RESULTS, f"seed_{seed}")
        result   = run_one_seed(seed, data, save_dir, split_seed=SPLIT_SEED)
        all_results.append(result)

        log(f"Seed {seed} done → "
            f"Acc={result['acc']:.4f}  Rec={result['rec']:.4f}  "
            f"Spec={result['spec']:.4f}  F1={result['f1']:.4f}  "
            f"ROC-AUC={result['roc_auc']:.4f}", "GOOD")
        print()

    sep('═')
    log("Computing ensemble probabilities (mean across 5 seeds) …")
    ens_labels = np.array(all_results[0]['t_labels_list'])
    ens_probs  = np.mean(
        [np.array(r['t_probs_list']) for r in all_results], axis=0)
    ens_cats   = all_results[0]['t_cats_list']

    # Youden threshold trên ensemble probs
    if len(set(ens_labels.tolist())) > 1:
        fpr_e, tpr_e, thr_e = roc_curve(ens_labels, ens_probs)
        ens_best_idx = int(np.argmax(tpr_e - fpr_e))
        ens_thr      = float(thr_e[ens_best_idx])
    else:
        ens_thr = INFERENCE_THR

    ens_roc_auc       = roc_auc_score(ens_labels, ens_probs) if len(set(ens_labels.tolist())) > 1 else 0.0
    pc_e, rc_e, _     = precision_recall_curve(ens_labels, ens_probs)
    ens_pr_auc        = auc(rc_e, pc_e) if len(set(ens_labels.tolist())) > 1 else 0.0
    ens_res           = eval_at_threshold(ens_labels, ens_probs, ens_thr)
    ens_cat_metrics   = eval_per_category(ens_labels, ens_probs, ens_cats, ens_thr)
    ens_cat_recall    = {cat: (ens_cat_metrics[cat]['rec'] if cat in ens_cat_metrics else float('nan'))
                         for cat in FALL_CATEGORIES}

    log(f"Ensemble threshold (Youden): {ens_thr:.4f}  "
        f"(Sensitivity={tpr_e[ens_best_idx]:.4f}  "
        f"Specificity={1-fpr_e[ens_best_idx]:.4f})", "GOOD")
    log(f"Ensemble → Acc={ens_res['acc']:.4f}  Rec={ens_res['rec']:.4f}  "
        f"Spec={ens_res['spec']:.4f}  F1={ens_res['f1']:.4f}  "
        f"ROC-AUC={ens_roc_auc:.4f}  PR-AUC={ens_pr_auc:.4f}", "GOOD")
    log("Ensemble per-category recall: " +
        "  ".join(f"{c.replace('_Falls','')}={ens_cat_recall[c]:.4f}" for c in FALL_CATEGORIES))

    ens_summary = dict(
        threshold  = ens_thr,
        roc_auc    = float(ens_roc_auc),
        pr_auc     = float(ens_pr_auc),
        cat_recall = {k: float(v) for k, v in ens_cat_recall.items()},
        **{k: float(v) if isinstance(v, (np.floating, float)) else int(v)
           for k, v in ens_res.items()},
    )
    with open(os.path.join(BASE_RESULTS, "ensemble_metrics.json"), "w") as f:
        json.dump(ens_summary, f, indent=2)
    log(f"Ensemble metrics saved → {BASE_RESULTS}/ensemble_metrics.json", "GOOD")

    sep('═')
    log("Generating stability plots …")
    os.makedirs(BASE_RESULTS, exist_ok=True)
    plot_stability_boxplot(all_results, BASE_RESULTS)
    plot_stability_per_category(all_results, BASE_RESULTS)
    plot_seed_comparison_bar(all_results, BASE_RESULTS)

    with open(os.path.join(BASE_RESULTS, "all_seeds_results.json"), "w") as f:
        json.dump(all_results, f, indent=2)
    summarize_all_seeds(all_results, BASE_RESULTS)
    print_stability_table(all_results)

    log(f"All done! Results in: {BASE_RESULTS}/", "GOOD")

def summarize_all_seeds(all_results: list, save_dir: str):
    """Ghi tổng kết mean±std của 5 seed vào summary_all_seeds.json"""

    scalar_keys = ['acc', 'prec', 'rec', 'spec', 'f1', 'mr', 'roc_auc', 'pr_auc']
    summary = {}

    # ── Global metrics ────────────────────────────────────────────────────────
    global_summary = {}
    for k in scalar_keys:
        vals = [r[k] for r in all_results]
        global_summary[k] = dict(
            mean   = float(np.mean(vals)),
            std    = float(np.std(vals)),
            values = [float(v) for v in vals],
        )
    summary['global_metrics'] = global_summary

    # ── Per-class metrics ─────────────────────────────────────────────────────
    per_class_summary = {}
    for cls in ['non_fall', 'fall']:
        cls_data = {}
        for metric in ['precision', 'recall', 'f1']:
            vals = [r['per_class_metrics'][cls][metric] for r in all_results]
            cls_data[metric] = dict(
                mean   = float(np.mean(vals)),
                std    = float(np.std(vals)),
                values = [float(v) for v in vals],
            )
        per_class_summary[cls] = cls_data
    summary['per_class_metrics'] = per_class_summary

    # ── Confusion matrices ────────────────────────────────────────────────────
    cms      = [r['confusion_matrix'] for r in all_results]
    cm_array = np.array(cms)
    summary['confusion_matrices'] = {
        f"seed_{r['seed']}": r['confusion_matrix'] for r in all_results
    }
    summary['confusion_matrix_aggregate'] = dict(
        mean = cm_array.mean(axis=0).tolist(),
        std  = cm_array.std(axis=0).tolist(),
        sum  = cm_array.sum(axis=0).tolist(),
    )

    # ── Per fall-category recall ───────────────────────────────────────────────
    cat_summary = {}
    for cat in FALL_CATEGORIES:
        vals = [r['cat_recall'].get(cat, float('nan')) for r in all_results]
        vals_clean = [v for v in vals if not np.isnan(v)]
        cat_summary[cat] = dict(
            mean   = float(np.mean(vals_clean)) if vals_clean else None,
            std    = float(np.std(vals_clean))  if vals_clean else None,
            values = [float(v) for v in vals],
        )
    summary['per_fall_category_recall'] = cat_summary

    # ── [NEW] Convergence history — mean±std theo epoch ───────────────────────
    #
    # Các seed có thể dừng sớm (early stopping) ở số epoch khác nhau.
    # Chiến lược: pad bằng giá trị cuối (last-value padding) để giữ
    # đường cong không bị đứt, sau đó tính mean/std trên toàn bộ seeds.
    # Lưu thêm "n_active" = số seed còn chạy tại epoch đó (chưa pad).
    #
    hist_keys = ['train_loss', 'val_loss', 'train_f2', 'val_f2']

    # Xác định độ dài tối đa
    max_len = max(len(r['history']['epoch']) for r in all_results)

    def pad_series(series: list, target_len: int) -> list:
        """Pad bằng giá trị cuối đến target_len."""
        if len(series) >= target_len:
            return series[:target_len]
        return series + [series[-1]] * (target_len - len(series))

    convergence = {}
    # Trục epoch chung (1-indexed, dùng epoch của seed dài nhất)
    convergence['epochs'] = list(range(1, max_len + 1))

    # n_active[i] = số seed thực sự có dữ liệu tại epoch i (không phải pad)
    n_active = []
    for ep_i in range(max_len):
        count = sum(1 for r in all_results if ep_i < len(r['history']['epoch']))
        n_active.append(count)
    convergence['n_active_seeds'] = n_active

    for key in hist_keys:
        # Ma trận (n_seeds × max_len) sau khi pad
        matrix = np.array([
            pad_series(r['history'][key], max_len)
            for r in all_results
        ], dtype=np.float64)   # shape: (n_seeds, max_len)

        mean_vals = matrix.mean(axis=0).tolist()
        std_vals  = matrix.std(axis=0).tolist()

        # lower/upper bound cho shaded area
        lower = (matrix.mean(axis=0) - matrix.std(axis=0)).tolist()
        upper = (matrix.mean(axis=0) + matrix.std(axis=0)).tolist()

        # Per-seed raw (để vẽ individual lines nếu muốn)
        per_seed = {
            f"seed_{r['seed']}": r['history'][key]
            for r in all_results
        }

        convergence[key] = dict(
            mean      = mean_vals,
            std       = std_vals,
            lower     = lower,    # mean - std  (shaded area bottom)
            upper     = upper,    # mean + std  (shaded area top)
            per_seed  = per_seed,
        )

    summary['convergence_history'] = convergence
    # ─────────────────────────────────────────────────────────────────────────

    summary['seeds']     = [r['seed'] for r in all_results]
    summary['n_seeds']   = len(all_results)
    summary['threshold_youden_mean'] = float(np.mean(
        [r.get('optimal_thr', INFERENCE_THR) for r in all_results]))

    path = os.path.join(save_dir, 'summary_all_seeds.json')
    with open(path, 'w') as f:
        json.dump(summary, f, indent=2)
    log(f"Summary saved → {path}", "GOOD")
    return summary
if __name__ == "__main__":
    main()


## Save Best Model Across Seeds


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL: Save Best Model Across All Seeds
# ══════════════════════════════════════════════════════════════════════════════
import shutil, json, os
import numpy as np

BASE_RESULTS = "results"          # phải khớp với hằng số trong script chính
BEST_MODEL_DIR  = os.path.join(BASE_RESULTS, "best_model_overall")
BEST_MODEL_PATH = os.path.join(BEST_MODEL_DIR, "best_model.pth")
BEST_META_PATH  = os.path.join(BEST_MODEL_DIR, "best_model_meta.json")
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

# ── 1. Đọc metrics của tất cả seed đã chạy ───────────────────────────────────
seed_dirs = sorted([
    d for d in os.listdir(BASE_RESULTS)
    if d.startswith("seed_") and
    os.path.isfile(os.path.join(BASE_RESULTS, d, "metrics.json"))
])

if not seed_dirs:
    raise FileNotFoundError(f"Không tìm thấy thư mục seed_* trong '{BASE_RESULTS}/'")

records = []
for sd in seed_dirs:
    meta_path  = os.path.join(BASE_RESULTS, sd, "metrics.json")
    model_path = os.path.join(BASE_RESULTS, sd, "best_model.pth")
    if not os.path.isfile(model_path):
        print(f"⚠️  Bỏ qua {sd}: không có best_model.pth")
        continue
    with open(meta_path) as f:
        m = json.load(f)
    records.append(dict(
        seed        = m['seed'],
        acc         = m['acc'],
        rec         = m['rec'],     # Sensitivity — quan trọng nhất với fall detection
        spec        = m['spec'],
        f1          = m['f1'],
        roc_auc     = m['roc_auc'],
        pr_auc      = m['pr_auc'],
        mr          = m['mr'],
        optimal_thr = m.get('optimal_thr', 0.65),
        model_path  = model_path,
        metrics_path= meta_path,
    ))

print(f"ℹ️  Tìm thấy {len(records)} seed có model:")
for r in records:
    print(f"    seed={r['seed']:>6}  Rec={r['rec']:.4f}  Spec={r['spec']:.4f}  "
          f"F1={r['f1']:.4f}  ROC-AUC={r['roc_auc']:.4f}")

# ── 2. Chọn seed tốt nhất theo composite score ────────────────────────────────
#   Score = 0.45·Recall + 0.25·Specificity + 0.20·F1 + 0.10·ROC-AUC
#   → Ưu tiên Sensitivity (không bỏ sót fall) nhưng vẫn kiểm soát FP
W_REC  = 0.45
W_SPEC = 0.25
W_F1   = 0.20
W_AUC  = 0.10

for r in records:
    r['score'] = (W_REC  * r['rec']  +
                  W_SPEC * r['spec'] +
                  W_F1   * r['f1']   +
                  W_AUC  * r['roc_auc'])

best = max(records, key=lambda x: x['score'])

print(f"\n✅ Seed tốt nhất: seed={best['seed']}  "
      f"(composite score={best['score']:.4f})")
print(f"   Rec={best['rec']:.4f}  Spec={best['spec']:.4f}  "
      f"F1={best['f1']:.4f}  ROC-AUC={best['roc_auc']:.4f}")
print(f"   Threshold (Youden): {best['optimal_thr']:.4f}")

# ── 3. Copy model & lưu metadata ─────────────────────────────────────────────
shutil.copy2(best['model_path'], BEST_MODEL_PATH)
print(f"\n✅ Đã lưu model → {BEST_MODEL_PATH}")

meta_out = dict(
    selected_seed   = best['seed'],
    composite_score = float(best['score']),
    weights         = dict(rec=W_REC, spec=W_SPEC, f1=W_F1, roc_auc=W_AUC),
    metrics         = {k: best[k] for k in ['acc','rec','spec','f1',
                                              'roc_auc','pr_auc','mr','optimal_thr']},
    all_seeds_scores= [
        dict(seed=r['seed'], score=round(r['score'],6),
             rec=r['rec'], spec=r['spec'], f1=r['f1'], roc_auc=r['roc_auc'])
        for r in sorted(records, key=lambda x: x['score'], reverse=True)
    ],
    source_path     = best['model_path'],
    saved_path      = BEST_MODEL_PATH,
)
with open(BEST_META_PATH, "w") as f:
    json.dump(meta_out, f, indent=2)
print(f"✅ Metadata lưu → {BEST_META_PATH}")

# ── 4. Hướng dẫn load lại model ──────────────────────────────────────────────
print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Cách load lại model tốt nhất:

    import torch
    from train_mydata_multiseed import ThreeStreamGSTCN

    model = ThreeStreamGSTCN().to(DEVICE)
    model.load_state_dict(
        torch.load("{BEST_MODEL_PATH}", weights_only=True)
    )
    model.eval()

    # Dùng threshold Youden tối ưu:
    BEST_THR = {best['optimal_thr']:.4f}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")


## SHAP Feature Importance


In [ ]:
# ============================================================
#  SHAP Feature Importance - Three-Stream STGCN (mydata)
#  Self-contained: no external script imports.
#  Output: results/shap/shap_results.json + 4 PNG figures
# ============================================================
# !pip install shap -q

import json
import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import shap
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")


## SHAP Config


In [ ]:
# ============================================================
#  A. CONFIG
# ============================================================
BEST_SEED = 42
MODEL_PATH = f"results/seed_{BEST_SEED}/best_model.pth"
NPZ_PATH = "/kaggle/input/datasets/toannv2020/mydata-fall-v2/custom_all.npz"
SAVE_DIR = "results/shap"

N_BACKGROUND = 80
N_EXPLAIN = 120
SHAP_CLASS = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(SAVE_DIR, exist_ok=True)

NUM_KEYPOINTS = 14
SEQ_LEN = 60
IN_CHANNELS = 3
NUM_CLASSES = 2

STREAM_CHANNELS = [24, 24, 24, 48, 48, 48, 96, 96, 96]
STREAM_STRIDES = [1, 1, 1, 2, 1, 1, 2, 1, 1]
TCN_KERNEL = 9
SKIP_BLOCK_IDX = 2
DROPOUT = 0.75
DROP_PATH_RATE = 0.18
CONF_THRESH = 0.5

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SPLIT_SEED = BEST_SEED

FALL_CATEGORIES = ["Lying_Falls", "Sitting_Falls", "Standing_Falls"]
ALL_CATEGORIES = ["non_fall"] + FALL_CATEGORIES
CATEGORY_LABEL = {
    "non_fall": 0,
    "Lying_Falls": 1,
    "Sitting_Falls": 2,
    "Standing_Falls": 3,
}

KEYPOINT_NAMES = [
    "Nose(0)", "LShoulder(1)", "RShoulder(2)", "LElbow(3)", "RElbow(4)",
    "LWrist(5)", "RWrist(6)", "LHip(7)", "RHip(8)",
    "LKnee(9)", "RKnee(10)", "LAnkle(11)", "RAnkle(12)", "MidHip(13)",
]
STREAM_NAMES = ["Joint", "Motion", "Angle"]

PARENT_MAP_AUG = {
    13: None, 7: 13, 8: 13, 1: 7, 2: 8, 0: 1,
    3: 1, 4: 2, 5: 3, 6: 4, 9: 7, 10: 8, 11: 9, 12: 10,
}


## SHAP Model Definition


In [ ]:
# ============================================================
#  B. MODEL DEFINITION - MATCH stgcn_3stream.py
# ============================================================
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = torch.empty(shape, dtype=x.dtype, device=x.device).bernoulli_(keep) / keep
        return x * mask


class SkeletonGraph:
    NUM_PARTITIONS = 3

    def __init__(self, num_nodes=NUM_KEYPOINTS):
        self.num_nodes = num_nodes
        self.neighbor_link = [
            (0, 1), (0, 2), (1, 2), (1, 3), (3, 5), (2, 4), (4, 6),
            (1, 7), (2, 8), (7, 8), (7, 9), (9, 11), (8, 10), (10, 12),
            (7, 13), (8, 13),
        ]
        self.HIP_DIST = {
            13: 0, 7: 1, 8: 1, 1: 2, 2: 2, 9: 2, 10: 2,
            0: 3, 3: 3, 4: 3, 11: 3, 12: 3, 5: 4, 6: 4,
        }
        self.A = self._build_adjacency()

    def _build_adjacency(self):
        K, V = self.NUM_PARTITIONS, self.num_nodes
        A = np.zeros((K, V, V), dtype=np.float32)

        for i in range(V):
            A[0, i, i] = 1.0

        for i, j in self.neighbor_link:
            di, dj = self.HIP_DIST[i], self.HIP_DIST[j]
            if di < dj:
                A[1, j, i] = 1
                A[2, i, j] = 1
            elif dj < di:
                A[1, i, j] = 1
                A[2, j, i] = 1
            else:
                A[1, i, j] = 1
                A[1, j, i] = 1

        for k in range(K):
            A[k] = self._sym_norm(A[k])
        return A

    @staticmethod
    def _sym_norm(A):
        A = A + np.eye(A.shape[0])
        d = A.sum(axis=1)
        d_inv = np.where(d > 0, d ** -0.5, 0.0)
        D = np.diag(d_inv)
        return D @ A @ D


class AdaptiveSpatialGraphConv(nn.Module):
    def __init__(self, in_ch, out_ch, num_nodes, num_partitions):
        super().__init__()
        self.K = num_partitions
        self.conv = nn.Conv2d(in_ch, out_ch * num_partitions, kernel_size=1)
        self.A_adapt = nn.Parameter(torch.zeros(num_partitions, num_nodes, num_nodes))

    def forward(self, x, A_static):
        A_eff = A_static + torch.tanh(self.A_adapt)
        N, _, T, V = x.shape
        x = self.conv(x).view(N, self.K, -1, T, V)
        return torch.einsum("kvw,nkctv->nctw", A_eff, x)


class TemporalConv(nn.Module):
    def __init__(self, channels, kernel_size=9, stride=1):
        super().__init__()
        pad = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(
            channels,
            channels,
            kernel_size=(kernel_size, 1),
            stride=(stride, 1),
            padding=(pad, 0),
        )
        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.tmp_maxpool = nn.MaxPool2d(
            kernel_size=(kernel_size, 1),
            stride=(stride, 1),
            padding=(pad, 0),
        )

    def forward(self, x):
        res = self.tmp_maxpool(x)
        x = self.relu(self.bn(self.conv(x)))
        return x + res


class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, num_nodes, num_partitions, stride=1, drop_path_rate=0.0):
        super().__init__()
        self.gcn = AdaptiveSpatialGraphConv(in_ch, out_ch, num_nodes, num_partitions)
        self.gcn_bn = nn.BatchNorm2d(out_ch)
        self.res_conv = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch))
            if in_ch != out_ch else nn.Identity()
        )
        self.sp_pool = nn.MaxPool2d((1, 3), (1, 1), (0, 1))
        self.tcn = TemporalConv(out_ch, TCN_KERNEL, stride)
        self.drop_path = DropPath(drop_path_rate)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, A):
        x_gcn = self.gcn_bn(self.gcn(x, A))
        x_res = self.relu(self.drop_path(x_gcn) + self.res_conv(x))
        x_sp = self.sp_pool(x_res)
        return self.relu(self.drop_path(self.tcn(x_sp)))


class StreamEncoder(nn.Module):
    def __init__(self, in_channels, adjacency, channels, strides):
        super().__init__()
        K, V, _ = adjacency.shape
        self.register_buffer("A", torch.tensor(adjacency, dtype=torch.float32))

        self.input_bn = nn.BatchNorm2d(in_channels)
        self.input_proj = nn.Sequential(
            nn.Conv2d(in_channels, channels[0], 1),
            nn.BatchNorm2d(channels[0]),
            nn.ReLU(inplace=True),
        )

        n_blocks = len(channels)
        blocks = []
        prev = channels[0]
        for i, (ch, st) in enumerate(zip(channels, strides)):
            dpr = DROP_PATH_RATE * i / max(1, n_blocks - 1)
            blocks.append(STGCNBlock(prev, ch, V, K, st, drop_path_rate=dpr))
            prev = ch

        self.blocks = nn.ModuleList(blocks)
        self.edge_importance = nn.ParameterList([
            nn.Parameter(torch.ones_like(self.A)) for _ in range(n_blocks)
        ])
        self.spatial_gap = nn.AdaptiveAvgPool2d((None, 1))

    def forward(self, x):
        x = self.input_proj(self.input_bn(x))
        early = None
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.A * self.edge_importance[i])
            if i == SKIP_BLOCK_IDX:
                early = x
        return self.spatial_gap(x).squeeze(-1), early


class CrossStreamSkip(nn.Module):
    def __init__(self, early_ch, final_ch):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(early_ch, final_ch, 1),
            nn.BatchNorm2d(final_ch),
            nn.ReLU(inplace=True),
        )
        self.spatial_gap = nn.AdaptiveAvgPool2d((None, 1))

    def forward(self, ej, em, ea):
        return self.spatial_gap(self.proj(ej + em + ea)).squeeze(-1)


class ConfidenceMasking(nn.Module):
    def __init__(self, conf_threshold=CONF_THRESH):
        super().__init__()
        self.threshold = conf_threshold

    def forward(self, feats, x_joint):
        T_prime = feats[0].shape[-1]
        conf = x_joint[:, 2, :, :].mean(dim=-1)
        conf_ds = F.adaptive_avg_pool1d(conf.unsqueeze(1).float(), T_prime).squeeze(1)
        mask = (conf_ds > self.threshold).float().unsqueeze(1)
        return [f * mask for f in feats], mask


class ClassificationModule(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, num_classes=2, dropout=0.5):
        super().__init__()
        self.fc1 = nn.Conv1d(in_dim, hidden_dim, 1)
        self.lrelu1 = nn.LeakyReLU(0.1, inplace=True)
        self.ln = nn.LayerNorm(hidden_dim)
        self.lrelu2 = nn.LeakyReLU(0.1, inplace=True)
        self.drop = nn.Dropout(p=dropout)
        self.fc_out = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.lrelu1(self.fc1(x))
        x = self.lrelu2(self.ln(x.permute(0, 2, 1)).permute(0, 2, 1))
        x = self.drop(x)
        x_avg = x.mean(dim=-1)
        return self.fc_out(x_avg)


class ThreeStreamSTGCN(nn.Module):
    def __init__(
        self,
        in_channels=IN_CHANNELS,
        num_classes=NUM_CLASSES,
        skeleton_adj=None,
        channels=STREAM_CHANNELS,
        strides=STREAM_STRIDES,
        dropout=DROPOUT,
    ):
        super().__init__()

        if skeleton_adj is None:
            skeleton_adj = SkeletonGraph().A

        self.stream_joint = StreamEncoder(in_channels, skeleton_adj, channels, strides)
        self.stream_motion = StreamEncoder(in_channels, skeleton_adj, channels, strides)
        self.stream_angle = StreamEncoder(in_channels, skeleton_adj, channels, strides)

        early_ch = channels[SKIP_BLOCK_IDX]
        final_ch = channels[-1]
        self.cross_skip = CrossStreamSkip(early_ch, final_ch)
        self.classifier = ClassificationModule(4 * final_ch, 128, num_classes, dropout)

    def forward(self, x_joint, x_motion, x_angle):
        f1, ej = self.stream_joint(x_joint)
        f2, em = self.stream_motion(x_motion)
        f3, ea = self.stream_angle(x_angle)

        T_prime = f1.shape[-1]
        f_skip = F.adaptive_avg_pool1d(self.cross_skip(ej, em, ea), T_prime)
        feat = torch.cat([f1, f2, f3, f_skip], dim=1)
        return self.classifier(feat)


## SHAP Data Helpers


In [ ]:
# ============================================================
#  C. DATA HELPERS
# ============================================================
def compute_motion_np(j):
    m = np.zeros_like(j)
    m[1:] = j[1:] - j[:-1]
    m[:, :, 2] = j[:, :, 2]
    return m


def compute_angle_np(j):
    Fv, V, _ = j.shape
    angle = np.zeros((Fv, V, 3), dtype=np.float32)

    for v in range(V):
        p = PARENT_MAP_AUG[v]
        if p is None:
            angle[:, v, 0] = 1.0
            angle[:, v, 2] = j[:, v, 2]
        else:
            dy = j[:, v, 1] - j[:, p, 1]
            dx = j[:, v, 0] - j[:, p, 0]
            th = np.arctan2(dy, dx)
            angle[:, v, 0] = np.cos(th)
            angle[:, v, 1] = np.sin(th)
            angle[:, v, 2] = np.minimum(j[:, v, 2], j[:, p, 2])

    return angle


def split_indices(
    y,
    subject_arr,
    is_flip_arr,
    category_arr,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    seed=42,
):
    rng = np.random.default_rng(seed)
    all_subjects = sorted(np.unique(subject_arr))
    all_cats = sorted(np.unique(category_arr))
    train_idx, val_idx, test_idx = [], [], []

    for subj in all_subjects:
        for cat in all_cats:
            mask_base = (
                (subject_arr == subj)
                & (category_arr == cat)
                & (is_flip_arr == 0)
            )
            base_ids = np.where(mask_base)[0]
            if len(base_ids) == 0:
                continue

            base_ids = base_ids[rng.permutation(len(base_ids))]
            n_total = len(base_ids)
            n_val = max(1, round(n_total * val_ratio))
            n_test = max(1, round(n_total * TEST_RATIO))
            n_test = min(n_test, n_total - n_val - 1)
            n_val = min(n_val, n_total - n_test - 1)
            n_train_base = n_total - n_val - n_test

            tr_base = base_ids[:n_train_base]
            va_ids = base_ids[n_train_base:n_train_base + n_val]
            te_ids = base_ids[n_train_base + n_val:]

            val_idx.extend(va_ids.tolist())
            test_idx.extend(te_ids.tolist())

            mask_flip = (
                (subject_arr == subj)
                & (category_arr == cat)
                & (is_flip_arr == 1)
            )
            flip_ids = np.where(mask_flip)[0]
            train_idx.extend(tr_base.tolist())
            train_idx.extend(flip_ids.tolist())

    return (
        np.array(train_idx, dtype=np.int64),
        np.array(val_idx, dtype=np.int64),
        np.array(test_idx, dtype=np.int64),
    )


def resolve_existing_path(primary, fallbacks):
    candidates = [primary] + list(fallbacks)
    for path in candidates:
        if os.path.exists(path):
            if path != primary:
                print(f"[WARN] Primary path not found: {primary}")
                print(f"[WARN] Using fallback: {path}")
            return path
    raise FileNotFoundError(
        "None of these paths exist:\n  " + "\n  ".join(candidates)
    )


def load_checkpoint_state(path):
    try:
        state = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        state = torch.load(path, map_location=DEVICE)

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    if any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}

    return state


def infer_stream_channels(state):
    if any(".sgc." in k or ".sep_tcn." in k for k in state.keys()):
        raise RuntimeError(
            "Checkpoint architecture mismatch: this file now matches stgcn_3stream.py "
            "(AdaptiveSpatialGraphConv + TemporalConv + CrossStreamSkip), but the "
            "checkpoint contains older SGC/Separable-TCN keys. Set MODEL_PATH to a "
            "checkpoint trained by the current stgcn_3stream.py, for example "
            f"results/seed_{BEST_SEED}/best_model.pth."
        )

    if any(k.startswith("masking.") for k in state.keys()):
        raise RuntimeError(
            "Checkpoint architecture mismatch: checkpoint contains the older "
            "ConfidenceMasking model, while this SHAP file now matches the "
            "CrossStreamSkip model in stgcn_3stream.py."
        )

    channels = []
    for i in range(len(STREAM_CHANNELS)):
        key = f"stream_joint.blocks.{i}.gcn.conv.weight"
        if key not in state:
            return STREAM_CHANNELS
        channels.append(int(state[key].shape[0] // SkeletonGraph.NUM_PARTITIONS))
    return channels


## SHAP Load Data


In [ ]:
# ============================================================
#  D. LOAD DATA AND MODEL
# ============================================================
NPZ_PATH = resolve_existing_path(NPZ_PATH, ["custom_all.npz"])
MODEL_PATH = resolve_existing_path(MODEL_PATH, ["best_model.pth"])

raw = np.load(NPZ_PATH, allow_pickle=True)
data = dict(
    X_joint=raw["X_joint"].astype(np.float32),
    y=raw["y"].astype(np.int64),
    img_w=raw["img_w"].astype(np.int32),
    img_h=raw["img_h"].astype(np.int32),
    subject=raw["subject"],
    is_flip=raw["is_flip"].astype(np.int32),
    category=raw["category"],
)

print(
    f"[OK] Data: {len(data['y'])} samples "
    f"(Fall={int(data['y'].sum())} NF={int((data['y'] == 0).sum())})"
)
print("     Categories:", {c: int((data["category"] == c).sum()) for c in ALL_CATEGORIES})

train_idx, val_idx, test_idx = split_indices(
    data["y"],
    data["subject"],
    data["is_flip"],
    data["category"],
    seed=SPLIT_SEED,
)
print(f"[OK] Split: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

state = load_checkpoint_state(MODEL_PATH)
checkpoint_channels = infer_stream_channels(state)
if checkpoint_channels != STREAM_CHANNELS:
    print(
        "[WARN] Checkpoint channels differ from stgcn_3stream.py defaults: "
        f"{checkpoint_channels} vs {STREAM_CHANNELS}"
    )
    print("[WARN] Building SHAP model with checkpoint channels so evaluation can run.")

model = ThreeStreamSTGCN(channels=checkpoint_channels).to(DEVICE)
model.load_state_dict(state, strict=True)
model.eval()
print(f"[OK] Model loaded: {MODEL_PATH}")


# ============================================================
#  E. TENSOR BUILDER
# ============================================================
def idx_to_tensors(indices):
    Js, Ms, As = [], [], []

    for i in indices:
        j = data["X_joint"][i].copy()
        m = compute_motion_np(j)
        a = compute_angle_np(j)

        def to_t(x):
            return torch.FloatTensor(x.transpose(2, 0, 1)).unsqueeze(0)

        Js.append(to_t(j))
        Ms.append(to_t(m))
        As.append(to_t(a))

    return (
        torch.cat(Js, 0).to(DEVICE),
        torch.cat(Ms, 0).to(DEVICE),
        torch.cat(As, 0).to(DEVICE),
    )


# ============================================================
#  F. BALANCED SAMPLE
# ============================================================
rng = np.random.default_rng(0)


def balanced_sample(pool, n_total):
    cats = data["category"][pool]
    out = []
    n_each = max(1, n_total // len(ALL_CATEGORIES))

    for cat in ALL_CATEGORIES:
        cands = pool[cats == cat]
        k = min(n_each, len(cands))
        if k > 0:
            out.extend(rng.choice(cands, k, replace=False).tolist())

    if len(out) < n_total:
        remain = np.setdiff1d(pool, np.array(out, dtype=np.int64), assume_unique=False)
        k = min(n_total - len(out), len(remain))
        if k > 0:
            out.extend(rng.choice(remain, k, replace=False).tolist())

    return np.array(out[:n_total], dtype=np.int64)


bg_idx = balanced_sample(test_idx, N_BACKGROUND)
exp_idx = balanced_sample(test_idx, N_EXPLAIN)

print(f"[OK] Background={len(bg_idx)} | Explain={len(exp_idx)}")
print("     Explain cat dist:", {c: int((data["category"][exp_idx] == c).sum()) for c in ALL_CATEGORIES})


# ============================================================
#  G. WRAPPER - CONCAT 3 STREAMS INTO ONE TENSOR
# ============================================================
C = IN_CHANNELS


class WrappedSTGCN(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base

    def forward(self, x_cat):
        j = x_cat[:, 0 * C:1 * C]
        m = x_cat[:, 1 * C:2 * C]
        a = x_cat[:, 2 * C:3 * C]
        return torch.softmax(self.base(j, m, a), dim=1)


wrapped = WrappedSTGCN(model).to(DEVICE)
wrapped.eval()

Tj_bg, Tm_bg, Ta_bg = idx_to_tensors(bg_idx)
Tj_exp, Tm_exp, Ta_exp = idx_to_tensors(exp_idx)
X_bg = torch.cat([Tj_bg, Tm_bg, Ta_bg], dim=1)
X_exp = torch.cat([Tj_exp, Tm_exp, Ta_exp], dim=1)


# ============================================================
#  H. SHAP GradientExplainer
# ============================================================
print("[INFO] Running SHAP GradientExplainer...")
explainer = shap.GradientExplainer(wrapped, X_bg)
shap_values = explainer.shap_values(X_exp)

raw_sv = np.array(shap_values) if not isinstance(shap_values, np.ndarray) else shap_values

if isinstance(shap_values, list):
    sv = np.array(shap_values[SHAP_CLASS])
elif raw_sv.ndim == 5 and raw_sv.shape[-1] == NUM_CLASSES:
    sv = raw_sv[..., SHAP_CLASS]
elif raw_sv.ndim == 5 and raw_sv.shape[0] == NUM_CLASSES:
    sv = raw_sv[SHAP_CLASS]
else:
    sv = raw_sv[..., SHAP_CLASS] if raw_sv.shape[-1] == NUM_CLASSES else raw_sv[SHAP_CLASS]

N_exp = len(exp_idx)
expected = N_exp * 9 * SEQ_LEN * NUM_KEYPOINTS
if sv.size != expected:
    raise RuntimeError(
        "[FAIL] SHAP shape parse error.\n"
        f"  raw_sv.shape={raw_sv.shape}, N_exp={N_exp}\n"
        f"  expected sv.size={expected}, got={sv.size}\n"
        f"  shap version: {shap.__version__}"
    )

sv = sv.reshape(N_exp, 9, SEQ_LEN, NUM_KEYPOINTS)
print(f"[OK] shap_values parsed: {sv.shape}")
sv_abs = np.abs(sv)


# ============================================================
#  I. AGGREGATE
# ============================================================
sv_4d = sv_abs.reshape(N_exp, 3, C, SEQ_LEN, NUM_KEYPOINTS)

stream_imp = sv_4d.mean(axis=(0, 2, 3, 4))
kp_imp = sv_4d.mean(axis=(0, 1, 2, 3))
temp_imp = sv_4d.mean(axis=(0, 1, 2, 4))
per_stream_kp = sv_4d.mean(axis=(0, 2, 3))

exp_cats = data["category"][exp_idx]
cat_kp = {}
for cat in ALL_CATEGORIES:
    mask = exp_cats == cat
    if mask.sum() > 0:
        cat_kp[cat] = sv_4d[mask].mean(axis=(0, 1, 2, 3))
    else:
        cat_kp[cat] = np.zeros(NUM_KEYPOINTS)

print("[OK] Aggregation done.")
print(f"  stream_imp shape : {stream_imp.shape}")
print(f"  kp_imp shape     : {kp_imp.shape}")
print(f"  temp_imp shape   : {temp_imp.shape}")


# ============================================================
#  J. SAVE JSON
# ============================================================
results_json = dict(
    method="GradientExplainer (SHAP)",
    reference="Lundberg & Lee, NeurIPS 2017",
    model="Three-Stream STGCN (mydata)",
    shap_class=SHAP_CLASS,
    n_background=int(len(bg_idx)),
    n_explain=int(len(exp_idx)),
    stream_importance={STREAM_NAMES[i]: float(stream_imp[i]) for i in range(3)},
    keypoint_importance={KEYPOINT_NAMES[i]: float(kp_imp[i]) for i in range(NUM_KEYPOINTS)},
    temporal_importance=[float(v) for v in temp_imp],
    per_stream_keypoint_importance={
        STREAM_NAMES[s]: {
            KEYPOINT_NAMES[v]: float(per_stream_kp[s, v])
            for v in range(NUM_KEYPOINTS)
        }
        for s in range(3)
    },
    per_category_keypoint_importance={
        cat: {KEYPOINT_NAMES[i]: float(v) for i, v in enumerate(vals)}
        for cat, vals in cat_kp.items()
    },
)

json_path = os.path.join(SAVE_DIR, "shap_results.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)
print(f"[OK] JSON: {json_path}")


# ============================================================
#  K. PLOTS
# ============================================================
kp_order = np.argsort(kp_imp)[::-1]
peak_t = int(np.argmax(temp_imp))

# K1 - Stream Importance
fig, ax = plt.subplots(figsize=(5, 3.5))
colors = ["#2171b5", "#e07b39", "#2ca02c"]
bars = ax.barh(STREAM_NAMES, stream_imp, color=colors, edgecolor="white", height=0.5)
for bar, val in zip(bars, stream_imp):
    ax.text(
        val + stream_imp.max() * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.6f}",
        va="center",
        fontsize=10,
    )
ax.set_xlabel("Mean |SHAP value|", fontsize=11)
ax.set_title("Stream-Level Importance (Fall class)\nGradientExplainer - SHAP", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "shap_stream_importance.png"), dpi=150)
plt.close(fig)

# K2 - Keypoint Importance
fig, ax = plt.subplots(figsize=(7, 6))
y_pos = np.arange(NUM_KEYPOINTS)
ax.barh(y_pos, kp_imp[kp_order], color="#2171b5", alpha=0.85, edgecolor="white")
ax.set_yticks(y_pos)
ax.set_yticklabels([KEYPOINT_NAMES[i] for i in kp_order], fontsize=9)
ax.set_xlabel("Mean |SHAP value|", fontsize=11)
ax.set_title("Keypoint Importance - All Streams Combined\nFall class - SHAP GradientExplainer", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "shap_keypoint_importance.png"), dpi=150)
plt.close(fig)

# K3 - Temporal Importance
fig, ax = plt.subplots(figsize=(9, 3.5))
t = np.arange(SEQ_LEN)
ax.fill_between(t, temp_imp, alpha=0.3, color="#9467bd")
ax.plot(t, temp_imp, color="#9467bd", lw=1.8)
ax.axvline(peak_t, color="#d62728", lw=1.4, ls="--", label=f"Peak frame t={peak_t}")
ax.set_xlabel("Frame index", fontsize=11)
ax.set_ylabel("Mean |SHAP|", fontsize=11)
ax.set_title("Temporal Importance - Fall class\nWhich frames matter most?", fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "shap_temporal_importance.png"), dpi=150)
plt.close(fig)

# K4 - Per-Category Heatmap
heat = np.array([cat_kp[c] for c in ALL_CATEGORIES])
heat_norm = heat / (heat.max(axis=1, keepdims=True) + 1e-12)

fig, ax = plt.subplots(figsize=(13, 3.5))
im = ax.imshow(heat_norm, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(NUM_KEYPOINTS))
ax.set_xticklabels(KEYPOINT_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(ALL_CATEGORIES)))
ax.set_yticklabels(ALL_CATEGORIES, fontsize=9)
ax.set_title("Per-Category Keypoint Importance (row-normalised)\nFall class - SHAP GradientExplainer", fontweight="bold")
plt.colorbar(im, ax=ax, fraction=0.012, pad=0.01, label="Relative importance")
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "shap_per_category_heatmap.png"), dpi=150, bbox_inches="tight")
plt.close(fig)


# ============================================================
#  L. CONSOLE SUMMARY
# ============================================================
print("\n" + "=" * 62)
print("  SHAP SUMMARY - Three-Stream STGCN - mydata")
print("=" * 62)

print("\n  Stream Importance:")
for s, v in zip(STREAM_NAMES, stream_imp):
    bar = "#" * int(v / stream_imp.max() * 24)
    print(f"    {s:<8}: {bar:<24}  {v:.6f}")

print("\n  Top-5 Keypoints (Fall class):")
for rank_i, ki in enumerate(kp_order[:5], 1):
    print(f"    #{rank_i}: {KEYPOINT_NAMES[ki]:<20} {kp_imp[ki]:.6f}")

print(f"\n  Peak temporal frame: t={peak_t}/{SEQ_LEN} ({peak_t / SEQ_LEN * 100:.1f}% into sequence)")

print("\n  Per fall-category top keypoint:")
for cat in FALL_CATEGORIES:
    top_k = int(np.argmax(cat_kp[cat]))
    print(f"    {cat:<20}: {KEYPOINT_NAMES[top_k]} ({cat_kp[cat][top_k]:.6f})")

print(f"\n  Saved to: {SAVE_DIR}/")
for fname in [
    "shap_results.json",
    "shap_stream_importance.png",
    "shap_keypoint_importance.png",
    "shap_temporal_importance.png",
    "shap_per_category_heatmap.png",
]:
    print(f"    {fname}")
print("=" * 62)
